In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import optuna
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import RidgeCV, ElasticNetCV, LassoCV
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
import re
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

import scipy.stats as scs
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.tsa.api as smt
from dateutil.relativedelta import relativedelta
from scipy.optimize import minimize
from tqdm.notebook import tqdm

# 1. Load Data

## 1.A. Load Data

In [ ]:
train_df = pd.read_csv('datatrain.csv')
test_df = pd.read_csv('datatest.csv')
submission_dates = test_df['date']
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

train_ids = train_df['obs_id']
test_ids = test_df['obs_id']
train_y = train_df['daily_ktCO2']

train_df = train_df.drop(columns=['daily_ktCO2', 'obs_id'])
test_df = test_df.drop(columns=['obs_id'])

In [ ]:
combined_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
combined_df.info()
combined_df

## 1.B. Convert to Appropriate Data Types

In [ ]:
# Change type of date col to datetime
combined_df['date'] = pd.to_datetime(combined_df['date'])
combined_df['date'].dtypes

In [ ]:
# There are some texts that are in lower case
text_cols = combined_df.select_dtypes(include=['object']).columns
for col in text_cols:
    if col != 'date':
        combined_df[col] = combined_df[col].astype(str).str.lower().str.strip()

In [ ]:
# Change bi-valued columns to "yes" or "no"
bool_cols = ['rain', 'weekend', 'holiday']
for col in bool_cols:
    if col in combined_df.columns:
        map_dict = {'true': "yes", 'yes': "yes", 'false': "no", 'no': "no"}
        combined_df[col] = combined_df[col].astype(str).str.strip().str.lower().map(map_dict)

for col in bool_cols:
    print(combined_df[col].unique())

In [ ]:
# Change so2 from object to numeric
combined_df['so2'] = pd.to_numeric(combined_df['so2'], errors='coerce')
combined_df['so2'].dtypes

# 2. Exploratory Data Analysis

In [ ]:
numerical_cols = [col for col in combined_df.columns if combined_df[col].dtype in [np.int64, np.float64]]
categorical_cols = [col for col in combined_df.columns if combined_df[col].dtype == 'object']

## 2.A. Time Series Visualization

In [ ]:
plot_y = pd.Series(np.nan, index=combined_df.index)
plot_y.iloc[:len(train_y)] = train_y.values
for col in numerical_cols:
    fig, ax1 = plt.subplots(figsize=(16,5))

    ax1.plot(combined_df['date'], combined_df[col], label=col, color='tab:blue', alpha=0.7)
    ax1.set_xlabel('Date')
    ax1.set_ylabel(f"{col}", color='tab:blue')
    ax1.tick_params(axis='y', labelcolor='tab:blue')
    missing_mask = combined_df[col].isna()
    ax1.plot(combined_df['date'][missing_mask], 
             [0]*missing_mask.sum(),
             'kx', label='Missing Data')
    
    ax2 = ax1.twinx()
    ax2.plot(combined_df['date'], plot_y, label='daily CO2', color='tab:red', alpha=0.7)
    ax2.set_ylabel("daily CO2", color='tab:red')
    ax2.tick_params(axis='y', labelcolor='tab:red')
    
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines + lines2, labels + labels2, loc="upper left")
    
    plt.title(f"{col} and Daily CO2 over the time")
    plt.show()

## 2.B. Histogram Distribution

In [ ]:
fig, axes = plt.subplots(len(numerical_cols) // 4, 4, figsize=(16, 4 * (len(numerical_cols) // 4)))
for ax, col in zip(axes.flatten(), numerical_cols):
    sns.histplot(x=col, data=train_df, ax=ax)
    ax.set_title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

## 2.C. Analysis of Numerical Features

Some columns contain missing values at different levels, ranging from only a few gaps to having most of the data missing.  
The approach to handling them depends on the pattern and amount of missing data.  

1. Columns with only a few missing values (isolated gaps)

- pm25  
- pm10 

$\rightarrow$ These can be imputed using **linear interpolation**, since the missing values are isolated single points between existing values.  

---

2. Columns with moderate missing values (scattered gaps)

- uvindex  
- moonphase  

$\rightarrow$ These can also be imputed using **linear interpolation**, because their time series patterns are oscillating and predictable between known points.  

---

3. Columns with moderate missing values (continuous blocks)

- TCI  
- o3  
- no2  

$\rightarrow$ These should be imputed using **time series forecasting methods**, as there is enough available data to train models and predict missing sequences.  

---

4. Columns with most values missing

- Car_7-9, Car_9-17, Car_17-19  
- Van_7-9, Van_9-17, Van_17-19  
- Bus_7-9, Bus_9-17, Bus_17-19  
- Truck_7-9, Truck_9-17, Truck_17-19  
- so2  

$\rightarrow$ These will be **dropped**, since the amount of missing data is too large to make meaningful predictions.


Some features also has outliers / suspiciously wrong value:

1. Columns with some values possibily recorded in different unit of measure:
* temp, tempmin, tempmax, dew, feelslike, feelslikemin, feelslikemax: recent datas are possibly recorded in Fahrenheit, not Celcius

$\rightarrow$ These should be **converted to celcius**

2. Columns with values out of range:
* moonphase: min and max value should be 0 and 1, but there are values less than 0 and more than 1
$\rightarrow$ all values should be capped at range [0, 1]

* uvindex: min value should be 0
$\rightarrow$ values less than 0 will be converted to the uv index mean of that week (uvindex 0 is not possible because that would mean no sunlight)

3. Columns with extreme right tail:
* precip
* pm10
$\rightarrow$ will be log-transformed to make data more normally distributed so that these large values will not so strongly influence the loss function

## 2.D. Categorical Visualization

In [ ]:
fig, axes = plt.subplots(len(categorical_cols)//2, 2,figsize=(14, 10))

for ax, col in zip(axes.flatten(), categorical_cols):
    sns.boxplot(x=col, y=train_y, data=combined_df.iloc[:len(train_y), :], ax=ax)
    ax.set_title(f'Boxplot of daily_ktCO2 by {col}')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

# 3. Data Cleaning

## 3.A. Transform Values

### 3.A.1. Fahrenheit to Celcius

In [ ]:
def f_to_c(df):
    temp_cols = [col for col in ['temp', 'feelslike', 'feelslikemax', 'feelslikemin', 'tempmax', 'tempmin', 'temp_calib', 'dew'] if col in df.columns]
    
    for col in temp_cols:
        if col in df.columns:
            df[col] = df[col].map(lambda x: (x-32) * (5/9) if x >= 60 else x)
    print(f"Transformed columns: {temp_cols}")
    
    for col in temp_cols:
        plt.figure(figsize=(14, 6))
        plt.plot(df['date'], df[col])
        plt.xlabel('Date')
        plt.ylabel(col)
        plt.title(f'{col} Over Time')
        plt.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
            
    return df

In [ ]:
combined_df = f_to_c(combined_df)

### 3.A.2. Values Out of Range

In [ ]:
def clip_moonphase(df):
    # moonphase range is only from 0 to 1
    df['moonphase'] = np.clip(df['moonphase'], 0, 1)
    
    plt.figure(figsize=(14, 6))
    plt.plot(df['date'], df['moonphase'])
    plt.xlabel('Date')
    plt.ylabel(col)
    plt.title('moonphase Over Time')
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    return df

In [ ]:
def transform_uvindex(df):
    weekly_mean = (
        df[df['uvindex'] > 0]
        .groupby(df['date'].dt.isocalendar().week)['uvindex']
        .transform('mean')
    )

    df.loc[df['uvindex'] <= 0, 'uvindex'] = weekly_mean[df['uvindex'] <= 0]

    plt.figure(figsize=(14, 6))
    plt.plot(df['date'], df['uvindex'])
    plt.xlabel('Date')
    plt.ylabel(col)
    plt.title('uvindex Over Time')
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    return df

In [ ]:
combined_df = clip_moonphase(combined_df)
combined_df = transform_uvindex(combined_df)

### 3.A.3. Extreme Right Tail

In [ ]:
def log_transform(df, cols):
    for col in cols:
        df[col] = np.log1p(df[col])
        sns.histplot(x=col, data=df)
        plt.title(f"Distribution of {col}")
        plt.show()
    return df

In [ ]:
combined_df = log_transform(combined_df, ['precip', 'pm10'])

## 3.B. Missing Values

### 3.B.1. Drop Columns

In [ ]:
def drop_columns(df, treshold):
    # Check missing columns

    missing_analysis = pd.DataFrame({
        'column': combined_df.columns,
        'missing_count': combined_df.isnull().sum(),
        'missing_pct': (combined_df.isnull().sum() / len(combined_df)) * 100
    })
    missing_analysis = missing_analysis[missing_analysis['missing_count'] > 0].sort_values('missing_pct', ascending=False)
    
    print("Columns with missing values:")
    print(missing_analysis)

    high_missing_cols = missing_analysis[missing_analysis['missing_pct'] > treshold]['column'].tolist()
    
    df = df.drop(columns=high_missing_cols)
    print(f"\nDropped {len(high_missing_cols)} columns with >{treshold}% missing data:")
    print(high_missing_cols)
    
    return df

In [ ]:
# Drop columns with >70% missing data
combined_df = drop_columns(combined_df, 70)

### 3.B.2. Imputation

In [ ]:
def fill_by_linear_interpolation(df, cols):
    for col in cols:
        df[col] = df[col].interpolate()

        plt.figure(figsize=(14, 6))
        plt.plot(df['date'], df[col])
        plt.xlabel('Date')
        plt.ylabel(col)
        plt.title(f'{col} Over Time')
        plt.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    return df

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import IterativeImputer
def fill_by_mice(X):
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    imp = IterativeImputer(
        estimator=rf,
        verbose=2,
        max_iter=5,
        tol=1e-10,
        imputation_order='roman'
    )
    ImputedData = imp.fit_transform(X)
    Imputed_data = pd.DataFrame(ImputedData, index=X.index, columns=X.columns)

    # plot each column with missing values highlighted
    for col in X.columns:
        missing_idx = X[X[col].isnull()].index
        
        plt.figure(figsize=[20, 6])
        plt.plot(X.index, Imputed_data[col], label=f"{col} (imputed)", color="blue")
        plt.scatter(missing_idx, Imputed_data.loc[missing_idx, col], color="red", label="Imputed points")
        
        plt.title(f"{col} with RandomForest Imputation")
        plt.ylabel(col)
        plt.xlabel("Time")
        plt.legend()
        plt.show()
    
    return Imputed_data

In [ ]:
def fill_by_doy_mean(df, cols):
    df["doy"] = df["date"].dt.dayofyear

    for col in cols:
        seasonal_avg = combined_df.groupby("doy")[col].mean()
        actual = df[col]
        df[col] = df.apply(
            lambda row: seasonal_avg[row["doy"]] if pd.isna(row[col]) else row[col],
            axis=1
        )

        plt.figure(figsize=(15, 7))
        plt.plot(df.index, actual, color="red", label='actual', alpha=1)
        plt.plot(df.index, df[col], color='blue', label='predicted', alpha=0.2)

    df = df.drop(['doy'], axis=1)
    return df

In [ ]:
combined_df = fill_by_linear_interpolation(combined_df, ['pm25', 'pm10', 'uvindex', 'moonphase', 'so2_imputed'])

In [ ]:
# X = combined_df.drop(['date'], axis=1)
# X_num = X.select_dtypes(include=['number'])
# X_num_imputed = fill_by_mice(X_num)
# combined_df[X_num.columns] = X_num_imputed

combined_df = fill_by_doy_mean(combined_df,['no2', 'o3', 'TCI', 'so2'])

In [ ]:
def impute_categorical_cols(df):
    categorical_cols = df.select_dtypes(include=['object'])
    for col in categorical_cols:
        # Convert literal "nan" strings into np.nan
        df[col] = df[col].replace("nan", np.nan)
    
        # If still has missing values, fill with mode (or 'unknown')
        if df[col].isnull().any():
            mode_vals = df[col].mode()
            fill_val = mode_vals[0] if len(mode_vals) > 0 else 'unknown'
            df[col] = df[col].fillna(fill_val)
    return df

In [ ]:
combined_df = impute_categorical_cols(combined_df)

In [ ]:
combined_df.isnull().sum()

# 4. Feature Engineering

## 4.A. Add Time Features

In [ ]:
def add_cyclical_features(df, col, period):
    """Add sine/cosine encoding for cyclical features"""
    if col in df.columns:
        df[f'{col}_sin'] = np.sin(2 * np.pi * df[col] / period)
        df[f'{col}_cos'] = np.cos(2 * np.pi * df[col] / period)
    return df

In [ ]:
def add_time_features(df):

    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    
    df['day_of_year'] = df['date'].dt.dayofyear
    df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
    df['day_of_week'] = df['date'].dt.dayofweek
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    
    df['quarter'] = df['date'].dt.quarter
    
    df['weekend'] = df['day_of_week'].apply(lambda x: 'yes' if x >= 5 else 'no')

    df = add_cyclical_features(df, 'day_of_week', 7)
    df = add_cyclical_features(df, 'day_of_year', 365.25)

    # cyclical_to_drop = ['year', 'month', 'quarter', 'week_of_year', 'day_of_week', 'day_of_year']
    # df = df.drop(columns=[col for col in cyclical_to_drop if col in df.columns])
    return df

In [ ]:
combined_df = add_time_features(combined_df)

## 4.B. Cyclical Encode Wind Direction

In [ ]:
combined_df = add_cyclical_features(combined_df, 'winddir', 360)

## 4.C. Add Feature Interactions

In [ ]:
def add_feature_interactions(df):
    df['temp_humidity'] = df['temp'] * df['humidity']
    df['wind_temp'] = df['temp'] * df['windspeed']
    df['pm_ratio'] = df['pm25'] / df['pm10']
    #df['temp_range'] = df['tempmax'] - df['tempmin']
    df['pressure_anomaly'] = df['sealevelpressure'] - df['sealevelpressure'].mean()
    df['no2_o3_ratio'] = df['no2'] / df['o3']
    
    solar_features = ['solarradiation', 'solarenergy', 'uvindex']
    for feat in solar_features:
        if feat in df.columns and 'cloudcover' in df.columns:
            df[f'{feat}_cloud_interaction'] = df[feat] * (100 - df['cloudcover']) / 100

    return df

In [ ]:
combined_df = add_feature_interactions(combined_df)

## 4.D. One-Hot Encode Categorical Variables

In [ ]:
def oh_encode(df):
    categorical_cols = df.select_dtypes(include=['object']).columns
    print(categorical_cols)
    df = pd.get_dummies(df, columns = categorical_cols, drop_first=True)
    return df

In [ ]:
combined_df = oh_encode(combined_df)

## 4.E. Drop Highly Correlated Feature

In [ ]:
def drop_highly_correlated(df, treshold):
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    if len(numerical_cols) > 1:
        corr_matrix = df[numerical_cols].corr().abs()
        upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        high_corr_features = [col for col in upper_tri.columns if any(upper_tri[col] > treshold)]
        if high_corr_features:
            df = df.drop(columns=high_corr_features)
            print(f"Removed {high_corr_features}")
    return df

## 4.F. Drop Similar Features

In [ ]:
def drop_similar_temp_features(df):
    cols_to_drop = ['dew', 'feelslikemax', 'feelslikemin', 'temp', 'windspeedmin', 'windspeedmax', 'windspeedmean', 'precipprob', 'precip', 'tempmax', 'tempmin']
    df = df.drop(cols_to_drop, axis=1)
    return df

In [ ]:
def drop_precipprob(df):
    return df.drop(['precipprob'], axis=1)

In [ ]:
combined_df = drop_precipprob(combined_df)

# 5. Model Building

In [ ]:
def root_mean_squared_error(ytrue, ypred):
    return np.sqrt(np.mean((ytrue-ypred)**2))

In [ ]:
combined_df = combined_df.drop(['month', 'day_of_week'], axis=1)

In [ ]:
len(combined_df)

In [ ]:
len(train_ids)

In [ ]:
combined_df.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in combined_df.columns]
combined_df.columns = [re.sub(r'_+', '_', col).strip('_') for col in combined_df.columns]

X_train = combined_df.iloc[train_ids-1, :].drop(['date'], axis=1)
X_test = combined_df.iloc[test_ids-1, :].drop(['date'], axis=1)

X_train.info()

In [ ]:
X_train.info()

In [ ]:
len(train_y)

In [ ]:
len(X_train)

In [ ]:
from optuna.samplers import TPESampler
sampler = TPESampler(seed=42)
# Cross-validation setup
NFOLDS = 5
tsfold = TimeSeriesSplit(n_splits=NFOLDS)

def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int("n_estimators", 100, 1000),
        'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int("num_leaves", 2, 256),
        'max_depth': trial.suggest_int("max_depth", -1, 50),
        'min_child_samples': trial.suggest_int("min_child_samples", 5, 100),
        'subsample': trial.suggest_float("subsample", 0.5, 1.0),
        'colsample_bytree': trial.suggest_float("colsample_bytree", 0.5, 1.0),
        'bagging_freq': trial.suggest_int("bagging_freq", 1, 10),
        'bagging_fraction': trial.suggest_float("bagging_fraction", 0.5, 1.0),
        'feature_fraction': trial.suggest_float("feature_fraction", 0.5, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        'random_state': 42,
        'verbose': -1
    }
    model = Pipeline([
        ('regressor', lgb.LGBMRegressor(**params))
    ])

    cv_scores = cross_val_score(model, X_train, train_y, cv=tsfold,
                                scoring='neg_root_mean_squared_error', n_jobs=-1)
    return -cv_scores.mean()


def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        'random_state': 42,
        'verbosity': 0
    }
    model = Pipeline([
        ('regressor', xgb.XGBRegressor(**params))
    ])
    cv_scores = cross_val_score(model, X_train, train_y, cv=tsfold,
                                scoring='neg_root_mean_squared_error', n_jobs=-1)
    return -cv_scores.mean()


def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int("iterations", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-5, 10.0, log=True),
        'verbose': 0,
        'random_state': 42
    }
    model = Pipeline([
        ('regressor', cb.CatBoostRegressor(**params))
    ])
    cv_scores = cross_val_score(model, X_train, train_y, cv=tsfold,
                                scoring='neg_root_mean_squared_error', n_jobs=-1)
    return -cv_scores.mean()

def objective_random_forest(trial):
    params = {
        'n_estimators': trial.suggest_int("n_estimators", 100, 1000),
        'max_depth': trial.suggest_int("max_depth", 5, 30),
        'min_samples_split': trial.suggest_int("min_samples_split", 2, 20),
        'min_samples_leaf': trial.suggest_int("min_samples_leaf", 1, 20),
        'max_features': trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        'random_state': 42,
        'n_jobs': -1
    }
    model = Pipeline([
        ('regressor', RandomForestRegressor(**params))
    ])
    cv_scores = cross_val_score(model, X_train, train_y, cv=tsfold,
                                scoring='neg_root_mean_squared_error', n_jobs=-1)
    return -cv_scores.mean()


def objective_extra_trees(trial):
    params = {
        'n_estimators': trial.suggest_int("n_estimators", 100, 1000),
        'max_depth': trial.suggest_int("max_depth", 5, 30),
        'min_samples_split': trial.suggest_int("min_samples_split", 2, 20),
        'min_samples_leaf': trial.suggest_int("min_samples_leaf", 1, 20),
        'max_features': trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        'random_state': 42,
        'n_jobs': -1
    }
    model = Pipeline([
        ('regressor', ExtraTreesRegressor(**params))
    ])
    cv_scores = cross_val_score(model, X_train, train_y, cv=tsfold,
                                scoring='neg_root_mean_squared_error', n_jobs=-1)
    return -cv_scores.mean()


def objective_gradient_boosting(trial):
    params = {
        'n_estimators':  trial.suggest_int("n_estimators", 100, 1000),
        'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int("max_depth", 3, 10),
        'random_state': 42
    }
    model = Pipeline([
        ('regressor', GradientBoostingRegressor(**params))
    ])
    cv_scores = cross_val_score(model, X_train, train_y, cv=tsfold,
                                scoring='neg_root_mean_squared_error', n_jobs=-1)
    return -cv_scores.mean()

    
study_lgb = optuna.create_study(direction="minimize", sampler=sampler)
study_lgb.optimize(lambda trial: objective_lgb(trial),
                   n_trials=25, show_progress_bar=True)

study_xgb = optuna.create_study(direction="minimize", sampler=sampler)
study_xgb.optimize(lambda trial: objective_xgb(trial),
                   n_trials=25, show_progress_bar=True)

study_cat = optuna.create_study(direction="minimize", sampler=sampler)
study_cat.optimize(lambda trial: objective_cat(trial),
                   n_trials=25, show_progress_bar=True)

study_rf = optuna.create_study(direction="minimize", sampler=sampler)
study_rf.optimize(lambda trial: objective_random_forest(trial),
                  n_trials=25, show_progress_bar=True)

study_et = optuna.create_study(direction="minimize", sampler=sampler)
study_et.optimize(lambda trial: objective_extra_trees(trial),
                  n_trials=25, show_progress_bar=True)

study_gb = optuna.create_study(direction="minimize", sampler=sampler)
study_gb.optimize(lambda trial: objective_gradient_boosting(trial),
                  n_trials=25, show_progress_bar=True)

In [ ]:
best_params = {
    "lightgbm": study_lgb.best_trial.params,
    "xgboost": study_xgb.best_trial.params,
    "catboost": study_cat.best_trial.params,
    "random_forest": study_rf.best_trial.params,
    "extra_trees": study_et.best_trial.params,
    'gradient_boosting': study_gb.best_trial.params
}

def get_models():
    """Return dictionary of models with optimized hyperparameters"""
    return {
        'lightgbm': lgb.LGBMRegressor(
            **best_params['lightgbm'],
            random_state=42, verbose=-1
        ),
        'xgboost': xgb.XGBRegressor(
            **best_params['xgboost'],
            random_state=42, verbosity=0
        ),
        'catboost': cb.CatBoostRegressor(
            **best_params['catboost'],
            random_state=42, verbose=0
        ),
        'random_forest': RandomForestRegressor(
            **best_params['random_forest'],
            random_state=42, n_jobs=-1
        ),
        'extra_trees': ExtraTreesRegressor(
            **best_params['extra_trees'],
            random_state=42, n_jobs=-1
        ),
        'gradient_boosting': GradientBoostingRegressor(
            **best_params['gradient_boosting'],
            random_state=42
        ),
        
        Linear models 
        'ridge': RidgeCV(alphas=np.logspace(-4, 4, 20), cv=tsfold),
        'elastic_net': ElasticNetCV(alphas=np.logspace(-4, 4, 10), 
                                   l1_ratio=[0.1, 0.5, 0.7, 0.9], cv=tsfold),
        'lasso': LassoCV(alphas=np.logspace(-4, 4, 20), cv=tsfold)
    }

# Single model evaluation
print("\n" + "="*60)
print("INDIVIDUAL MODEL PERFORMANCE")
print("="*60)

models = get_models()
model_scores = {}
model_predictions = {}

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    
    cv_scores = cross_val_score(model, X_train, train_y, cv=tsfold, 
                               scoring='neg_root_mean_squared_error', n_jobs=-1)
    cv_rmse = -cv_scores.mean()
    cv_std = cv_scores.std()
    
    model_scores[name] = {'cv_rmse': cv_rmse, 'cv_std': cv_std}
    
    model.fit(X_train, train_y)
    test_pred = model.predict(X_test)
    model_predictions[name] = test_pred
    
    print(f"  CV RMSE: {cv_rmse:.4f} (±{cv_std:.4f})")

print(f"\n{'Model':<15} {'CV RMSE':<10} {'Std':<10}")
print("-" * 35)
for name, scores in sorted(model_scores.items(), key=lambda x: x[1]['cv_rmse']):
    print(f"{name:<15} {scores['cv_rmse']:<10.4f} {scores['cv_std']:<10.4f}")

In [ ]:
def plot_feature_importance(model, X_train, title):
    """
        Plots sorted coefficient values of the model
    """
    importance_scores = model.feature_importances_
    scores = pd.DataFrame(importance_scores, X_train.columns)
    scores.columns = ["importance"]
    scores = scores.sort_values(by="importance", ascending=False)

    plt.figure(figsize=(15, 7))
    scores.importance.plot(kind="bar")
    plt.grid(True, axis="y")
    plt.hlines(y=0, xmin=0, xmax=len(scores), linestyles="dashed")
    plt.title(title)
    plt.show();

In [ ]:
# -----------------------------------------------------------------
# 6. Advanced Ensemble Methods
# -----------------------------------------------------------------
print("\n" + "="*60)
print("ENSEMBLE METHODS")
print("="*60)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# OOF prediction untuk stacking
print("\nGenerating out-of-fold predictions...")
selected_model_names = [name for name, _ in sorted(model_scores.items(), key=lambda x: x[1]['cv_rmse'])[:5]]
print(f"Selected top 5 models: {selected_model_names}")

oof_predictions = np.zeros((len(X_train), len(selected_model_names)))
test_predictions = np.zeros((len(X_test), len(selected_model_names)))

models_for_stacking = get_models()

for i, name in enumerate(selected_model_names):
    print(f"--- Stacking training for {name} ---")
    model = models_for_stacking[name]
    fold_test_preds = []
    # evaluating the training process
    for fold, (train_ids, val_ids) in enumerate(tsfold.split(X_train)):
        X_fold_train, X_fold_val = X_train.iloc[train_ids], X_train.iloc[val_ids]
        y_fold_train, y_fold_val = train_y.iloc[train_ids], train_y.iloc[val_ids]

        model.fit(X_fold_train, y_fold_train)
        plot_feature_importance(model, X_fold_train, f"Feature Importance of Model {name} of Fold {fold}")
        val_pred = model.predict(X_fold_val)
        oof_predictions[val_ids, i] = val_pred

    # testing (using several folds of training data) (USING THIS IMPROVES SCORE FROM 1.943 TO 1.841)
    for fold, (train_ids, val_ids) in enumerate(kfold.split(X_train)):
        X_fold_train, X_fold_val = X_train.iloc[train_ids], X_train.iloc[val_ids]
        y_fold_train, y_fold_val = train_y.iloc[train_ids], train_y.iloc[val_ids]

        model.fit(X_fold_train, y_fold_train)
        test_pred = model.predict(X_test)
        fold_test_preds.append(test_pred)
    test_predictions[:, i] = np.mean(fold_test_preds, axis=0)

# -----------------------------------------------------------------
# 7. Meta-Model Training and Final Prediction
# -----------------------------------------------------------------
print("\nTraining meta-models...")
meta_models = {
    'ridge_meta': RidgeCV(alphas=np.logspace(-4, 4, 20)),
    'elastic_meta': ElasticNetCV(alphas=np.logspace(-4, 4, 10), 
                                l1_ratio=[0.1, 0.5, 0.7, 0.9])
}

best_meta_rmse = float('inf')
best_meta_model = None
best_meta_name = None

mask = np.all(oof_predictions == 0, axis=1)
count = np.sum(mask)
train_y_t = train_y[count:]
oof_predictions_t = oof_predictions[count:]
for meta_name, meta_model in meta_models.items():
    meta_model.fit(oof_predictions_t, train_y_t)
    meta_pred = meta_model.predict(oof_predictions_t)
    meta_rmse = np.sqrt(mean_squared_error(train_y_t, meta_pred))
    
    print(f"{meta_name}: {meta_rmse:.4f}")
    
    if meta_rmse < best_meta_rmse:
        best_meta_rmse = meta_rmse
        best_meta_model = meta_model
        best_meta_name = meta_name

print(f"\nBest meta-model: {best_meta_name} (RMSE: {best_meta_rmse:.4f})")

final_predictions = best_meta_model.predict(test_predictions)
final_predictions = np.maximum(final_predictions, 0)

In [ ]:
# -----------------------------------------------------------------
# 8. Create Submission
# -----------------------------------------------------------------
print("\nStep 8: Creating Submission...")

submission_df = pd.DataFrame({
    'date': submission_dates,
    'daily_ktCO2': np.array(final_predictions)
})

submission_df.to_csv('submission.csv', index=False)
print("Submission saved to 'submission.csv'")
print(f"\nFirst 5 predictions:")
print(submission_df.head())

print(f"\nTraining completed! Final OOF RMSE: {best_meta_rmse:.4f}")